# Part F: final strengthening analyses

Run this **after** v5 (`log2_loggenes`), Part D and Part E. It only **reads** their outputs
(`pancreatic_paper_checkpoints_log2_loggenes/`, `.../partD/`, `.../partE/`) and writes everything new to
`.../partF/`. Every slow step checkpoints, so after a disconnect just **Run all** again.

| Cell | What it does | CPU time (approx.) |
|---|---|---|
| F0-F0b | setup, helpers, loads v5 / Part D / Part E results | 2 min |
| F1 | **Equivalence tests** (90% CIs of each method minus IG vs a stated margin, plus the smallest margin at which equivalence holds) for every faithfulness metric and planted-gene recovery | 5 min |
| F2 | **Moffitt subtypes, fixed**: continuous basal-like score with median and tertile splits (replaces the failed k-means split) | seconds |
| F3 | **Disagreement decomposition with interactions**, and the decomposition among converged explanations only (low-budget SHAP removed) | 5 min |
| F4 | **Second and third architectures**: concatenation-fusion network (all 8 methods) and random survival forest (model-agnostic methods; data-randomisation sanity check) | 50 min |
| F5 | **PDAC-only cohort (n = 157)**: the key Table 5 metrics in-sample and out-of-fold | 35 min |
| F6 | **More clinical variables** (sex, grade, residual disease, nodal status from the GDC): prediction and exact modality shares | 10 min |
| F7 | summary JSON | seconds |

**First run with `QUICK_RUN = True`** (about 15 minutes, writes to `partF_quick/`, numbers NOT for the paper), then set
`QUICK_RUN = False` and Run all.

In [ ]:
# ===== F0. Setup =====
import os, sys, subprocess, json, pickle, time, copy, warnings, urllib.request, re
from itertools import combinations, product
from math import factorial

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lifelines', 'shap', 'lime', 'captum',
                    'scikit-survival'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')

QUICK_RUN = False     # True = ~15-minute dry run with tiny budgets (numbers NOT for the paper). Set False for the real run.
BASE      = os.environ.get('PAPER_BASE', '/content/drive/MyDrive')
DATA_DIR  = f'{BASE}/pancreatic_paper_data'
V5_DIR    = f'{BASE}/pancreatic_paper_checkpoints_log2_loggenes'
PARTD_DIR = f'{V5_DIR}/partD'
PARTE_DIR = f'{V5_DIR}/partE' if os.path.isdir(f'{V5_DIR}/partE') else f'{V5_DIR}/partE_quick'
OUT       = f'{V5_DIR}/partF' + ('_quick' if QUICK_RUN else '')
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
import requests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr, rankdata, wilcoxon
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lime.lime_tabular import LimeTabularExplainer
from captum.attr import IntegratedGradients, GradientShap, DeepLift
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
warnings.filterwarnings('ignore')
torch.set_num_threads(max(1, os.cpu_count() or 1))

SHAP_LO     = 200
SHAP_HI     = 256 if QUICK_RUN else 4096
GS_SAMPLES  = 10 if QUICK_RUN else 50
N_BOOT      = 200 if QUICK_RUN else 1000
N_RANDOM    = 5 if QUICK_RUN else 20
EPOCHS      = 30 if QUICK_RUN else 300
PFI_REPEATS = 2 if QUICK_RUN else 10
RSF_TREES   = 30 if QUICK_RUN else 300
RSF_N       = 10 if QUICK_RUN else 40      # patients explained by KernelSHAP / LIME on the forest (cost)
RSF_BG      = 10 if QUICK_RUN else 20      # KernelSHAP background size for the forest
MARGIN      = {'del_auc': 0.01, 'ins_auc': 0.01, 'comp': 0.05, 'suff': 0.05, 'AUPRC': 0.02}   # equivalence margins


def save(name, obj):
    path = f'{OUT}/{name}.pkl'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


def load(name):
    path = f'{OUT}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def load_from(folder, name):
    path = f'{folder}/{name}.pkl'
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def load_v5(name):
    return load_from(V5_DIR, name)


RESULTS = load('results') or {}


def record(key, value):
    RESULTS[key] = value
    save('results', RESULTS)


METHODS  = ['SHAP-200', 'SHAP-4096', 'LIME', 'IG', 'GradSHAP', 'DeepLIFT', 'PFI', 'PDP']
LOCAL_M  = ['SHAP-200', 'SHAP-4096', 'LIME', 'IG', 'GradSHAP', 'DeepLIFT']
pd.set_option('display.width', 250); pd.set_option('display.max_colwidth', 70); pd.set_option('display.max_columns', 40)
print(f"QUICK_RUN = {QUICK_RUN}; reading Part E from {PARTE_DIR}; outputs -> {OUT}")

In [ ]:
# ===== F0b. Data, models and helpers (same code as Part E) =====
DATA_CSV = f'{DATA_DIR}/tcga_paad_clean_log2.csv'
df = pd.read_csv(DATA_CSV)
stage_map = {'STAGE I': 1, 'STAGE IA': 1, 'STAGE IB': 1, 'STAGE II': 2, 'STAGE IIA': 2, 'STAGE IIB': 2,
             'STAGE III': 3, 'STAGE IV': 4}
df['STAGE_ENCODED'] = df['AJCC_PATHOLOGIC_TUMOR_STAGE'].map(stage_map).fillna(2)
gene_cols = df.columns[5:1005].tolist()
CLIN_COLS = ['AGE', 'STAGE_ENCODED']
NC = len(CLIN_COLS)
ID_COL = next(c for c in df.columns[:5] if df[c].astype(str).str.startswith('TCGA-').all())
PID = df[ID_COL].astype(str).str[:12].values
N = len(df)
f32 = np.float32

Xc_raw = df[CLIN_COLS].values.astype(f32)
Xg_log = df[gene_cols].values.astype(f32)
if Xg_log.max() > 100:
    Xg_log = np.log2(Xg_log + 1)


def clip_pct(A, ref=None):
    ref = A if ref is None else ref
    return np.clip(A, np.percentile(ref, 1, axis=0), np.percentile(ref, 99, axis=0)).astype(f32)


Xg_clip = clip_pct(Xg_log)
T = df['OS_MONTHS'].values.astype(f32)
E = df['OS_STATUS'].values.astype(f32)
E_int = E.astype(int)
FEATURES = CLIN_COLS + gene_cols
FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xc_raw, E))
print(f"{N} patients, {len(gene_cols)} genes, {int(E.sum())} deaths")


def cox_ph_loss(risk, T_, E_):
    order = torch.argsort(T_, descending=True)
    risk, E_ = risk[order], E_[order]
    return -torch.mean((risk - torch.logcumsumexp(risk, dim=0)) * E_)


class GatedFusion(nn.Module):
    kind = 'fusion'

    def __init__(self, n_clin=2, n_gen=1000):
        super().__init__()
        self.clin_encoder = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                          nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gen_encoder = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4))
        self.gate_clin = nn.Sequential(nn.Linear(192, 64), nn.Sigmoid())
        self.gate_gen = nn.Sequential(nn.Linear(192, 128), nn.Sigmoid())
        self.head = nn.Linear(192, 1)

    def forward(self, xc, xg):
        hc, hg = self.clin_encoder(xc), self.gen_encoder(xg)
        h = torch.cat([hc, hg], 1)
        return self.head(torch.cat([hc * self.gate_clin(h), hg * self.gate_gen(h)], 1)).squeeze(1)


GatedFusionModel = GatedFusion     # the v5 checkpoints were saved under this class name


def _forward(model, xc, xg):
    return model(xc, xg)


def train_model(model, Xc, Xg, T_, E_, epochs=None, lr=1e-3):
    epochs = epochs or EPOCHS
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xc, Xg = torch.tensor(Xc, dtype=torch.float32), torch.tensor(Xg, dtype=torch.float32)
    Tt, Et = torch.tensor(T_, dtype=torch.float32), torch.tensor(E_, dtype=torch.float32)
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = cox_ph_loss(model(Xc, Xg), Tt, Et)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sch.step()
    return model.eval()


def risk_of(model, Xc, Xg):
    with torch.no_grad():
        return model(torch.tensor(Xc, dtype=torch.float32), torch.tensor(Xg, dtype=torch.float32)).numpy()


class Flat(nn.Module):
    """f([clinical | genes]) for the explainers."""
    def __init__(self, base, nc=NC):
        super().__init__()
        self.base, self.nc = base, nc

    def forward(self, x):
        return self.base(x[:, :self.nc], x[:, self.nc:])


class LinearFlat(nn.Module):
    """A linear Cox model f(x) = beta . x, for the analytic ground truth."""
    def __init__(self, beta):
        super().__init__()
        self.lin = nn.Linear(len(beta), 1, bias=False)
        with torch.no_grad():
            self.lin.weight.copy_(torch.tensor(np.asarray(beta, dtype=f32))[None])

    def forward(self, x):
        return self.lin(x).squeeze(1)


def np_fn(module):
    def fn(X):
        with torch.no_grad():
            return module(torch.tensor(np.asarray(X, dtype=f32))).numpy()
    return fn


def harrell_c_many(T_, E_, R):
    """Harrell's C for every row of R; same pairing and tie rules as lifelines.concordance_index(T, -risk, E)."""
    T_ = np.asarray(T_, float); E_ = np.asarray(E_).astype(bool); R = np.atleast_2d(np.asarray(R, float))
    comp = ((T_[:, None] < T_[None, :]) | ((T_[:, None] == T_[None, :]) & ~E_[None, :])) & E_[:, None]
    n = comp.sum()
    if n == 0:
        return np.full(len(R), np.nan)
    out = np.empty(len(R))
    for s in range(0, len(R), 64):                     # chunks keep memory small
        Rc = R[s:s + 64]
        gt = (Rc[:, :, None] > Rc[:, None, :]) & comp
        eq = (Rc[:, :, None] == Rc[:, None, :]) & comp
        out[s:s + 64] = (gt.sum((1, 2)) + 0.5 * eq.sum((1, 2))) / n
    return out


def harrell_c(T_, E_, r):
    return float(harrell_c_many(T_, E_, np.asarray(r)[None])[0])


# ---- explainers (identical settings to v5 for SHAP-200, LIME, IG) ----
def ig_attr(module, X, baseline_vec, n_steps=50):
    X_t = torch.tensor(np.asarray(X, dtype=f32), requires_grad=True)
    base = torch.tensor(np.tile(np.asarray(baseline_vec, dtype=f32), (len(X), 1)))
    return IntegratedGradients(module).attribute(X_t, baselines=base, n_steps=n_steps,
                                                 internal_batch_size=4096).detach().numpy()


def kernel_shap(fn, X, bg_pool, nsamples, seed=42, l1='num_features(50)', n_bg=30):
    """v5 KernelSHAP: background = shap.sample(pool, 30) (random_state 0); `seed` sets the coalition sampling."""
    np.random.seed(seed)
    ex = shap.KernelExplainer(fn, shap.sample(bg_pool, min(n_bg, len(bg_pool))))
    sv = ex.shap_values(np.asarray(X, dtype=f32), nsamples=nsamples, l1_reg=l1, silent=True)
    return np.asarray(sv).reshape(len(X), -1)


def lime_local(fn, X_explain, X_train, num_samples=300, seed=42):
    ex = LimeTabularExplainer(training_data=X_train, mode='regression', discretize_continuous=False, random_state=seed)
    out = np.zeros(X_explain.shape, dtype=float)
    for i in range(len(X_explain)):
        e = ex.explain_instance(X_explain[i], fn, num_features=X_explain.shape[1], num_samples=num_samples)
        for j, w in e.as_map()[1]:
            out[i, j] = w
    return out


def gradshap_attr(module, X, background, n_samples=None, seed=42):
    """GradientSHAP (expected gradients) with the same 30-patient background as KernelSHAP."""
    n_samples = n_samples or GS_SAMPLES
    torch.manual_seed(seed)
    B = torch.tensor(np.asarray(background, dtype=f32))
    out = []
    for s in range(0, len(X), 32):
        Xb = torch.tensor(np.asarray(X[s:s + 32], dtype=f32), requires_grad=True)
        out.append(GradientShap(module).attribute(Xb, baselines=B, n_samples=n_samples, stdevs=0.0).detach().numpy())
    return np.concatenate(out)


def deeplift_attr(module, X, baseline_vec):
    """DeepLIFT (Rescale rule) from the same median baseline as IG. Returns attributions and completeness gaps."""
    X_t = torch.tensor(np.asarray(X, dtype=f32), requires_grad=True)
    base = torch.tensor(np.tile(np.asarray(baseline_vec, dtype=f32), (len(X), 1)))
    a, d = DeepLift(module).attribute(X_t, baselines=base, return_convergence_delta=True)
    return a.detach().numpy(), d.detach().numpy()


def pfi_global(fn, X, T_, E_, n_repeats=None, seed=42):
    """Permutation importance: mean C-index drop when a feature is permuted (vectorised over repeats)."""
    n_repeats = n_repeats or PFI_REPEATS
    rng = np.random.RandomState(seed)
    c0 = harrell_c(T_, E_, fn(X))
    imp = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        Xs = np.repeat(X[None], n_repeats, axis=0)
        for r in range(n_repeats):
            Xs[r, :, j] = X[rng.permutation(len(X)), j]
        preds = fn(Xs.reshape(-1, X.shape[1])).reshape(n_repeats, len(X))
        imp[j] = c0 - harrell_c_many(T_, E_, preds).mean()
    return imp


def pdp_global(fn, X, cand, n_grid=10):
    out = np.zeros(X.shape[1])
    for j in cand:
        grid = np.percentile(X[:, j], np.linspace(10, 90, n_grid))
        Xs = np.repeat(X[None], n_grid, axis=0)
        Xs[np.arange(n_grid), :, j] = grid[:, None]
        curve = fn(Xs.reshape(-1, X.shape[1])).reshape(n_grid, len(X)).mean(1)
        out[j] = curve.max() - curve.min()
    return out


def randomise(model, seed=0):
    torch.manual_seed(seed)
    rm = copy.deepcopy(model)
    for mod in rm.modules():
        if isinstance(mod, nn.Linear):
            mod.reset_parameters()
    return rm.eval()


def top_genes(g, names=FEATURES, k=20, nc=NC):
    o = np.argsort(-np.asarray(g))
    return [names[i] for i in o if i >= nc][:k]


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(1, len(a | b))


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj


class ClinicalMLP(nn.Module):
    def __init__(self, n_clin=2, n_gen=None):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                 nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, xc, xg=None):
        return self.net(xc).squeeze(1)


class ConcatFusion(nn.Module):
    def __init__(self, n_clin=2, n_gen=1000):
        super().__init__()
        self.clin_encoder = nn.Sequential(nn.Linear(n_clin, 32), nn.BatchNorm1d(32), nn.ReLU(),
                                          nn.Linear(32, 64), nn.BatchNorm1d(64), nn.ReLU())
        self.gen_encoder = nn.Sequential(nn.Linear(n_gen, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                                         nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.4))
        self.head = nn.Linear(192, 1)

    def forward(self, xc, xg):
        return self.head(torch.cat([self.clin_encoder(xc), self.gen_encoder(xg)], 1)).squeeze(1)


def vs_ref_wilcoxon(W, ref='IG'):
    """W: DataFrame (rows = patients, cols = methods). Holm-corrected Wilcoxon of every method vs `ref`."""
    others = [m for m in W.columns if m != ref]
    p = [wilcoxon(W[m], W[ref]).pvalue if np.any(W[m] != W[ref]) else 1.0 for m in others]
    return {m: float(a) for m, a in zip(others, holm(p))} if p else {}


def vs_ref_boot(boot, ref='IG'):
    others = [m for m in boot if m != ref]
    p = []
    for m in others:
        d = boot[m] - boot[ref]; d = d[np.isfinite(d)]
        p.append(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())) if len(d) else np.nan)
    return {m: float(a) for m, a in zip(others, holm(p))} if p else {}


def faith_table(point, boot, label, methods, ref='IG'):
    p_del = vs_ref_boot({m: boot['del_auc'][:, i] for i, m in enumerate(methods)}, ref)
    p_ins = vs_ref_boot({m: boot['ins_auc'][:, i] for i, m in enumerate(methods)}, ref)
    rows = []
    for i, m in enumerate(methods):
        rows.append({'method': m, 'setting': label, 'del_auc': point['del_auc'][i],
                     'del_lo': np.nanpercentile(boot['del_auc'][:, i], 2.5), 'del_hi': np.nanpercentile(boot['del_auc'][:, i], 97.5),
                     'del_p_vs_ref': p_del.get(m, np.nan), 'ins_auc': point['ins_auc'][i],
                     'ins_lo': np.nanpercentile(boot['ins_auc'][:, i], 2.5), 'ins_hi': np.nanpercentile(boot['ins_auc'][:, i], 97.5),
                     'ins_p_vs_ref': p_ins.get(m, np.nan)})
    return pd.DataFrame(rows)


def exact_group_shapley(fn, X, groups, background):
    names = list(groups); k = len(names); n = len(X)

    def v(S):
        out = np.zeros(n)
        for brow in background:
            Xs = np.tile(brow, (n, 1)).astype(f32)
            for g in S:
                Xs[:, groups[g]] = X[:, groups[g]]
            out += fn(Xs)
        return out / len(background)

    vals = {S: v(S) for r in range(k + 1) for S in combinations(names, r)}
    phi = {}
    for g in names:
        others = [h for h in names if h != g]
        tot = np.zeros(n)
        for r in range(k):
            for S in combinations(others, r):
                w = factorial(r) * factorial(k - r - 1) / factorial(k)
                tot += w * (vals[tuple(sorted(S + (g,), key=names.index))] - vals[S])
        phi[g] = tot
    return pd.DataFrame(phi)


def genomic_share_exact(fn, X, nc, n_bg=60):
    bg = X[np.random.RandomState(0).choice(len(X), min(n_bg, len(X)), replace=False)]
    phi = exact_group_shapley(fn, X, {'clinical': list(range(nc)), 'genomic': list(range(nc, X.shape[1]))}, bg).abs()
    return float((phi['genomic'] / (phi.sum(1) + 1e-12)).mean())


def summed_share(A, nc):
    A = np.abs(A)
    return float((A[:, nc:].sum(1) / (A.sum(1) + 1e-12)).mean())


def boot_diff_ci(a, b, n_boot=None, seed=0, alpha=0.10):
    """Paired bootstrap of mean(a - b): point, (1-alpha) CI."""
    n_boot = n_boot or N_BOOT
    d = np.asarray(a, float) - np.asarray(b, float); d = d[np.isfinite(d)]
    rb = np.random.RandomState(seed)
    bs = np.array([d[rb.randint(0, len(d), len(d))].mean() for _ in range(n_boot)])
    return d.mean(), np.percentile(bs, 100 * alpha / 2), np.percentile(bs, 100 * (1 - alpha / 2))


def fit_rsf(X, T_, E_, seed=42):
    return RandomSurvivalForest(n_estimators=RSF_TREES, min_samples_leaf=10, max_features='sqrt', random_state=seed,
                                n_jobs=-1, low_memory=True).fit(np.asarray(X, float), Surv.from_arrays(np.asarray(E_).astype(bool), T_))


KS = [1, 2, 5, 10, 20, 50, 100]
KS_LOCAL = [5, 10, 20]
FILLS = ['median', 'mean', 'resample']
N_DRAWS = 2 if QUICK_RUN else 5


def filled_pred(fn, X, cols, mode, ref, rng):
    """Model output when columns `cols` of X are removed with the given fill-in (ref = reference patients)."""
    draws = N_DRAWS if mode == 'resample' else 1
    out = 0.0
    for _ in range(draws):
        Xp = X.copy()
        if len(cols):
            if mode == 'median':
                Xp[:, cols] = np.median(ref, 0)[cols]
            elif mode == 'mean':
                Xp[:, cols] = ref.mean(0)[cols]
            else:
                Xp[:, cols] = ref[rng.randint(0, len(ref), len(X))][:, cols]
        out = out + fn(Xp)
    return out / draws


def make_stack(fn, X, orders, mode, ref, seed=42):
    """Predictions for deletion / insertion of each method's top-k and of random k-sets; last row = unmodified."""
    rs = np.random.RandomState(seed); rng = np.random.RandomState(seed + 1)
    p = X.shape[1]; allc = np.arange(p)
    rnd = [[rs.choice(p, k, replace=False) for k in KS] for _ in range(N_RANDOM)]
    rows = []
    for m in orders:
        rows += [filled_pred(fn, X, orders[m][:k], mode, ref, rng) for k in KS]
    for m in orders:
        rows += [filled_pred(fn, X, np.setdiff1d(allc, orders[m][:k]), mode, ref, rng) for k in KS]
    for r in rnd:
        rows += [filled_pred(fn, X, ix, mode, ref, rng) for ix in r]
    for r in rnd:
        rows += [filled_pred(fn, X, np.setdiff1d(allc, ix), mode, ref, rng) for ix in r]
    rows.append(fn(X))
    return np.stack(rows)


def stack_stats(T_, E_, S, n_m, ii):
    K_ = len(KS)
    C = harrell_c_many(T_[ii], E_[ii], S[:, ii])
    a, b, c = n_m * K_, 2 * n_m * K_, 2 * n_m * K_ + N_RANDOM * K_
    cd, ci = C[:a].reshape(n_m, K_), C[a:b].reshape(n_m, K_)
    rd, ri = C[b:c].reshape(N_RANDOM, K_), C[c:-1].reshape(N_RANDOM, K_)
    c0 = C[-1]
    drop = c0 - cd
    return {'drop': drop, 'ins': ci, 'del_auc': drop.mean(1) - (c0 - rd).mean(), 'ins_auc': ci.mean(1) - ri.mean(),
            'rnd_drop': (c0 - rd).mean(0), 'rnd_ins': ri.mean(0), 'c0': c0}


def evaluate_stacks(parts, n_m, seed=7):
    """parts: list of (T, E, STACK) - one per fold (or one for the whole cohort). Point estimate = mean over parts;
    CIs from resampling patients within each part."""
    def stats(idx_list):
        ss = [stack_stats(T_, E_, S, n_m, ii) for (T_, E_, S), ii in zip(parts, idx_list)]
        return {k: np.nanmean([s[k] for s in ss], axis=0) for k in ss[0]}
    point = stats([np.arange(len(T_)) for T_, _, _ in parts])
    rb = np.random.RandomState(seed)
    boots = [stats([rb.randint(0, len(T_), len(T_)) for T_, _, _ in parts]) for _ in range(N_BOOT)]
    return point, {k: np.array([b[k] for b in boots]) for k in ('del_auc', 'ins_auc')}


def comp_suff(fn, X, attr_by_method, mode, ref, sd):
    """Per-patient comprehensiveness / sufficiency (risk change in units of sd), mean over k = 5, 10, 20."""
    rng = np.random.RandomState(3)
    r0 = fn(X)
    comp, suff = {}, {}
    for m, A in attr_by_method.items():
        imp = np.abs(A) if A.ndim == 2 else np.tile(A, (len(X), 1))
        c_acc, s_acc = np.zeros(len(X)), np.zeros(len(X))
        for k in KS_LOCAL:
            top = np.argsort(-imp, axis=1)[:, :k]
            draws = N_DRAWS if mode == 'resample' else 1
            pd_, ps_ = 0.0, 0.0
            for _ in range(draws):
                if mode == 'median':
                    fillrow = np.tile(np.median(ref, 0), (len(X), 1))
                elif mode == 'mean':
                    fillrow = np.tile(ref.mean(0), (len(X), 1))
                else:
                    fillrow = ref[rng.randint(0, len(ref), len(X))].copy()
                Xd = X.copy(); np.put_along_axis(Xd, top, np.take_along_axis(fillrow, top, 1), axis=1)
                Xk = fillrow.copy(); np.put_along_axis(Xk, top, np.take_along_axis(X, top, 1), axis=1)
                pd_ = pd_ + fn(Xd); ps_ = ps_ + fn(Xk)
            c_acc += np.abs(r0 - pd_ / draws) / sd
            s_acc += np.abs(r0 - ps_ / draws) / sd
        comp[m], suff[m] = c_acc / len(KS_LOCAL), s_acc / len(KS_LOCAL)
    return pd.DataFrame(comp), pd.DataFrame(suff)


# ---- saved v5 / Part D / Part E results ----
def need(folder, name):
    obj = load_from(folder, name)
    if obj is None:
        raise FileNotFoundError(f"{name}.pkl not found in {folder}. Run v5, Part D and Part E first.")
    return obj


cv_state = need(V5_DIR, 'cv_state')
FOLD_MODELS = [m.eval() for m in cv_state['fold_fusion_models']]
FOLD_SCALERS = cv_state['fold_scalers']
assert all(np.array_equal(np.sort(te), np.sort(t5)) for (tr, te), t5 in zip(FOLDS, cv_state['fold_test_indices']))
_ms = need(V5_DIR, 'model_full_state')
model_full = GatedFusion(); model_full.load_state_dict(_ms['model_full_state_dict']); model_full.eval()
X_all = _ms['X_all'].astype(f32)
F_full = Flat(model_full).eval(); pred_full = np_fn(F_full)
MED_ALL = np.median(X_all, axis=0).astype(f32)
BG30 = shap.sample(X_all, 30)
OOF5 = need(V5_DIR, 'extended_eval_oof_preds')
hist_csv = f'{PARTD_DIR}/D2_patient_histology_purity.csv'
HIST = pd.read_csv(hist_csv).set_index('PATIENT_ID').reindex(PID) if os.path.exists(hist_csv) else None
PDAC = HIST['histology'].isin(['ductal', 'adenocarcinoma NOS']).values if HIST is not None else None
print(f"Loaded v5 model and folds; PDAC subset: {None if PDAC is None else int(PDAC.sum())} patients; Part E: {PARTE_DIR}")

In [ ]:
# ===== F1. Equivalence tests: is each method practically equivalent to IG? =====
# Two one-sided tests (TOST) at alpha = 0.05 are equivalent to asking whether the 90% CI of (method - IG) lies inside
# +/- margin. Margins (MARGIN in F0): 0.01 C-index for deletion / insertion AUC; 0.05 risk-SD units for per-patient
# comprehensiveness / sufficiency; 0.02 for planted-gene AUPRC. Because any margin is a choice, we also report the
# SMALLEST margin at which equivalence would hold (max |90% CI bound|) - readers can apply their own.
def eq_row(metric, setting, method, point, lo, hi, margin):
    return {'metric': metric, 'setting': setting, 'method': method, 'diff_vs_IG': point, 'ci90_lo': lo, 'ci90_hi': hi,
            'margin': margin, 'equivalent': bool(lo > -margin and hi < margin),
            'smallest_equiv_margin': max(abs(lo), abs(hi))}


rows = []
iIG = METHODS.index('IG')
# (a) deletion / insertion AUC: paired bootstrap replicates saved by Part E (in-sample E3, out-of-fold E4)
for setting, name in [('in-sample, median', 'E3_median'), ('in-sample, mean', 'E3_mean'),
                      ('in-sample, resample', 'E3_resample'), ('OOF, median', 'E4_delins_median'),
                      ('OOF, mean', 'E4_delins_mean')]:
    ck = need(PARTE_DIR, name)
    for metric in ['del_auc', 'ins_auc']:
        B = ck['boot'][metric]
        for i, m in enumerate(METHODS):
            if m == 'IG':
                continue
            d = B[:, i] - B[:, iIG]; d = d[np.isfinite(d)]
            lo, hi = np.percentile(d, [5, 95])
            rows.append(eq_row(metric, setting, m, ck['point'][metric][i] - ck['point'][metric][iIG], lo, hi, MARGIN[metric]))

# (b) per-patient comprehensiveness / sufficiency: in-sample (E3) and out-of-fold (recomputed from Part E's fold explanations)
def fold_arrays(f):
    tr, te = FOLDS[f]
    sc, sg = FOLD_SCALERS[f]
    X_tr = np.hstack([sc.transform(Xc_raw[tr]), sg.transform(Xg_clip[tr])]).astype(f32)
    X_te = np.hstack([sc.transform(Xc_raw[te]), sg.transform(Xg_clip[te])]).astype(f32)
    return tr, te, X_tr, X_te


ck = load('F1_oof_comp_suff')
if ck is None:
    comp_all, suff_all = [], []
    for f in range(5):
        tr, te, X_tr, X_te = fold_arrays(f)
        fe = need(PARTE_DIR, f'E4_fold{f}')
        fn = np_fn(Flat(FOLD_MODELS[f]).eval())
        attrs = {**{m: fe['local'][m] for m in LOCAL_M}, 'PFI': fe['PFI'], 'PDP': fe['PDP']}
        cp, sf = comp_suff(fn, X_te, attrs, 'median', X_tr, fn(X_tr).std())
        comp_all.append(cp); suff_all.append(sf)
    ck = {'comp': pd.concat(comp_all, ignore_index=True), 'suff': pd.concat(suff_all, ignore_index=True)}
    save('F1_oof_comp_suff', ck)
sources = [('in-sample, median', need(PARTE_DIR, 'E3_median')), ('in-sample, resample', need(PARTE_DIR, 'E3_resample')),
           ('OOF, median', ck)]
for setting, src in sources:
    for metric in ['comp', 'suff']:
        W = src[metric]
        for m in METHODS:
            if m == 'IG':
                continue
            pnt, lo, hi = boot_diff_ci(W[m].values, W['IG'].values)
            rows.append(eq_row(metric, setting, m, pnt, lo, hi, MARGIN[metric]))

# (c) planted-gene AUPRC: paired over simulation replicates
syn = pd.DataFrame(need(PARTE_DIR, 'E7_planted'))
for eff in sorted(syn.effect.unique()):
    W = syn[syn.effect == eff].pivot(index='seed', columns='method', values='AUPRC')
    for m in METHODS:
        if m == 'IG' or m not in W:
            continue
        pnt, lo, hi = boot_diff_ci(W[m].values, W['IG'].values)
        rows.append(eq_row('AUPRC', f'planted, strength {eff:g} ({len(W)} reps)', m, pnt, lo, hi, MARGIN['AUPRC']))

F1 = pd.DataFrame(rows)
print(F1.round(4).to_string(index=False))
summary = F1.pivot_table(index='method', columns=['metric', 'setting'], values='equivalent', aggfunc='first').reindex(
    [m for m in METHODS if m != 'IG'])
print("\nEquivalent to IG at the stated margin (True / False):")
print(summary.to_string())
print("\nNote: 'not equivalent' is not the same as 'worse' - check the sign of diff_vs_IG and the CI.")
F1.to_csv(f'{OUT}/F1_equivalence.csv', index=False)
summary.to_csv(f'{OUT}/F1_equivalence_summary.csv')
record('F1_equivalence', F1.to_dict('records'))

In [ ]:
# ===== F2. Moffitt subtypes, fixed: continuous basal-like score with median and tertile splits =====
# Part E's k-means split put 166 / 177 patients in one group, so its labels (and that log-rank p) are discarded.
# The continuous score from Part E (mean z of 25 basal-like minus 25 classical genes) is used directly.
sub = pd.read_csv(f'{PARTE_DIR}/E8_patient_subtypes.csv')
assert list(sub.PATIENT_ID.astype(str).str[:12]) == list(PID), "Patient order differs"
score = sub['basal_score'].values
oof_r = np.empty(N)
for f, (tr, te) in enumerate(FOLDS):
    oof_r[te] = OOF5['Gated fusion']['te'][f]
cohorts = {'All 177': np.ones(N, bool)}
if PDAC is not None:
    cohorts['PDAC histology'] = PDAC
rows = []
fig, axes = plt.subplots(1, len(cohorts), figsize=(5.2 * len(cohorts), 4), squeeze=False)
for ax, (name, msk) in zip(axes[0], cohorts.items()):
    s, t, e = score[msk], T[msk], E[msk]
    hi = s > np.median(s)
    tert = np.digitize(s, np.quantile(s, [1 / 3, 2 / 3]))
    lr_med = logrank_test(t[hi], t[~hi], e[hi], e[~hi]).p_value
    lr_ter = multivariate_logrank_test(t, tert, e).p_value
    d = pd.DataFrame({'T': t, 'E': e, 'basal_score_per_SD': (s - s.mean()) / s.std(),
                      'oof_risk_per_SD': (oof_r[msk] - oof_r[msk].mean()) / oof_r[msk].std()})
    c1 = CoxPHFitter().fit(d[['T', 'E', 'basal_score_per_SD']], 'T', 'E')
    c2 = CoxPHFitter().fit(d, 'T', 'E')
    rows.append({'cohort': name, 'n': int(msk.sum()),
                 'logrank_p_median_split': lr_med, 'logrank_p_tertiles': lr_ter,
                 'HR_per_SD_basal_score': c1.hazard_ratios_['basal_score_per_SD'], 'p_basal_score': c1.summary.loc['basal_score_per_SD', 'p'],
                 'C_basal_score': harrell_c(t, e.astype(int), s), 'C_network_oof': harrell_c(t, e.astype(int), oof_r[msk]),
                 'C_score_plus_network': harrell_c(t, e.astype(int), c2.predict_log_partial_hazard(d).values),
                 'HR_network_adj': c2.hazard_ratios_['oof_risk_per_SD'], 'p_network_adj': c2.summary.loc['oof_risk_per_SD', 'p'],
                 'HR_score_adj': c2.hazard_ratios_['basal_score_per_SD'], 'p_score_adj': c2.summary.loc['basal_score_per_SD', 'p'],
                 'spearman_score_vs_network': spearmanr(s, oof_r[msk]).correlation})
    for g, lab, col in [(0, 'most classical third', '#2a78d6'), (1, 'middle third', '#8a8a85'), (2, 'most basal-like third', '#eb6834')]:
        KaplanMeierFitter().fit(t[tert == g], e[tert == g], label=f'{lab} (n={int((tert == g).sum())})').plot_survival_function(ax=ax, ci_show=False, color=col)
    ax.set_title(f'{name}: tertiles of the basal-like score (log-rank p = {lr_ter:.3g})', fontsize=9)
    ax.set_xlabel('Overall survival (months)'); ax.set_ylabel('Survival probability'); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_moffitt_km.png', dpi=400, bbox_inches='tight'); plt.close()
F2 = pd.DataFrame(rows)
print(F2.round(4).to_string(index=False))
print("C_score_plus_network is in-sample for the Cox combination (optimistic); use it only to compare with each alone.")
F2.to_csv(f'{OUT}/F2_moffitt_fixed.csv', index=False)
record('F2_moffitt', F2.to_dict('records'))

In [ ]:
# ===== F3. Disagreement decomposition: interactions, and converged explanations only =====
# Part D's main-effects model explained only 8-17% of the variance in pairwise disagreement. Here we ask (a) how much
# two-way interactions add, and which; and (b) what the decomposition looks like once the noisy low-budget KernelSHAP
# explanations (SHAP-200) are removed, i.e. among IG, LIME and SHAP-2048 only.
GRIDP = {'transform': ['raw', 'log2'], 'selection': ['variance', 'mad', 'cox'], 'k': [500, 1000, 2000],
         'scaler': ['robust', 'standard']}
if not os.path.exists(f'{PARTD_DIR}/grid'):
    raise FileNotFoundError("Part D grid checkpoints not found")
ex_rows = []
for v in product(*GRIDP.values()):
    p = dict(zip(GRIDP, v))
    key = f"{p['transform']}_{p['selection']}_k{p['k']}_{p['scaler']}"
    rec = load_from(f'{PARTD_DIR}/grid', key)
    if rec is None:
        continue
    for (s, fam, budget, run) in rec['expl']:
        ex_rows.append({**p, 'pipe': key, 'seed': s, 'family': fam, 'budget': budget, 'run': run})
EX = pd.DataFrame(ex_rows)
PAIRS = load_from(PARTD_DIR, 'D4_pairs')
assert PAIRS is not None and len(PAIRS) == len(EX) * (len(EX) - 1) // 2, "D4_pairs does not match the grid"
FACT = ['transform', 'selection', 'k', 'scaler', 'model seed', 'method family', 'SHAP budget', 'SHAP re-run']


def r2(y, X):
    A = np.column_stack([np.ones(len(y)), X])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    return 1 - np.sum((y - A @ beta) ** 2) / np.sum((y - y.mean()) ** 2)


def shapley_r2(y, X, names):
    p = len(names); cache = {}

    def R(S):
        if S not in cache:
            cache[S] = r2(y, X[:, list(S)]) if S else 0.0
        return cache[S]
    out = np.zeros(p)
    for f in range(p):
        others = [g for g in range(p) if g != f]
        for k_ in range(p):
            for S in combinations(others, k_):
                w = factorial(k_) * factorial(p - k_ - 1) / factorial(p)
                out[f] += w * (R(tuple(sorted(S + (f,)))) - R(S))
    return out, R(tuple(range(p)))


rows_int, rows_conv = [], []
for metric, lab in [('d_rho', '1 - Spearman'), ('d_jac', '1 - top-20 Jaccard')]:
    ok = PAIRS[metric].notna().values
    P_ = PAIRS[ok]
    y = P_[metric].values
    Xm = P_[FACT].values.astype(float)
    # (a) interactions
    inter_names = [f'{a} x {b}' for a, b in combinations(FACT, 2)]
    Xi = np.column_stack([Xm[:, i] * Xm[:, j] for i, j in combinations(range(len(FACT)), 2)])
    keep = Xi.std(0) > 0
    Xi, inter_names = Xi[:, keep], [n for n, k_ in zip(inter_names, keep) if k_]
    r2_main, r2_full = r2(y, Xm), r2(y, np.column_stack([Xm, Xi]))
    drops = []
    for t in range(Xi.shape[1]):
        drops.append(r2_full - r2(y, np.column_stack([Xm, np.delete(Xi, t, axis=1)])))
    top = np.argsort(drops)[::-1][:6]
    rows_int.append({'metric': lab, 'R2_main_effects': r2_main, 'R2_main_plus_2way': r2_full,
                     'top_interactions (unique R2)': '; '.join(f'{inter_names[t]} ({drops[t]:.3f})' for t in top)})
    # (b) converged explanations only: drop every pair involving a SHAP-200 explanation
    lo_ex = ((EX.family == 'SHAP') & (EX.budget == SHAP_LO)).values
    conv = ~lo_ex[P_['i'].values] & ~lo_ex[P_['j'].values]
    yc, Xc_ = y[conv], P_.loc[conv, FACT[:6]].values.astype(float)
    sh, r2c = shapley_r2(yc, Xc_, FACT[:6])
    single = Xc_.sum(1) == 1
    for f, name in enumerate(FACT[:6]):
        sel = single & (Xc_[:, f] == 1)
        rows_conv.append({'metric': lab, 'factor': name, 'n_single_factor_pairs': int(sel.sum()),
                          'one_factor_mean_disagreement': yc[sel].mean() if sel.any() else np.nan,
                          'shapley_R2': sh[f], 'share_of_explained_%': 100 * sh[f] / sh.sum(), 'total_R2': r2c})
F3a, F3b = pd.DataFrame(rows_int), pd.DataFrame(rows_conv)
print("(a) Main effects vs main effects + all two-way interactions:")
print(F3a.round(3).to_string(index=False))
print("\n(b) Converged explanations only (IG, LIME, SHAP-2048; SHAP-200 removed):")
print(F3b.round(3).to_string(index=False))
F3a.to_csv(f'{OUT}/F3_interactions.csv', index=False); F3b.to_csv(f'{OUT}/F3_converged_decomposition.csv', index=False)
record('F3', {'interactions': F3a.to_dict('records'), 'converged': F3b.to_dict('records')})

In [ ]:
# ===== F4. Does the picture hold for other architectures? =====
# (a) Concatenation-fusion network (no gates): all 8 methods, the same metrics as Table 5 (in-sample).
# (b) Random survival forest (non-differentiable): KernelSHAP-200 (two runs), KernelSHAP-4096, LIME, PFI, PDP;
#     sanity check by DATA randomisation (forest refit on permuted survival labels; Adebayo et al. 2018).
# Questions: does low-budget KernelSHAP stay noise-dominated, do converged methods agree, does summing |attributions|
# still inflate the genomic share, and is any method clearly more faithful?
def explain_all(F, X, ref, T_, E_, tag, hi=True):
    """All 8 methods for a differentiable model F on patients X; `ref` = reference patients (background / LIME data)."""
    ck = load(f'F_{tag}_expl') or {}
    fn = np_fn(F); med = np.median(ref, 0).astype(f32); bg = shap.sample(ref, 30)
    jobs = {'SHAP-200': lambda: kernel_shap(fn, X, ref, SHAP_LO, seed=42),
            'SHAP-200b': lambda: kernel_shap(fn, X, ref, SHAP_LO, seed=43),
            'LIME': lambda: lime_local(fn, X, ref, num_samples=300, seed=42),
            'IG': lambda: ig_attr(F, X, med),
            'GradSHAP': lambda: gradshap_attr(F, X, bg, seed=42),
            'DeepLIFT': lambda: deeplift_attr(F, X, med)[0]}
    if hi:
        jobs['SHAP-4096'] = lambda: kernel_shap(fn, X, ref, SHAP_HI, seed=42, l1=False)
    for m, job in jobs.items():
        if m not in ck:
            t0 = time.time(); ck[m] = job(); ck[f't_{m}'] = time.time() - t0
            save(f'F_{tag}_expl', ck)
    if 'PFI' not in ck:
        ck['PFI'] = pfi_global(fn, X, T_, E_); save(f'F_{tag}_expl', ck)
    if 'PDP' not in ck:
        g = {m: np.abs(ck[m]).mean(0) for m in ['SHAP-200', 'LIME', 'IG']}
        cand = sorted(set().union(*[set(np.argsort(-v)[:25]) for v in g.values()], set(np.argsort(-ck['PFI'])[:25])))
        ck['PDP'] = pdp_global(fn, X, cand); ck['PDP_cand'] = cand; save(f'F_{tag}_expl', ck)
    return ck


def gi_of(ck, methods):
    return {m: (np.abs(ck[m]).mean(0) if np.ndim(ck[m]) == 2 else ck[m]) for m in methods if m in ck}


def agreement(GI_, ck, ref_m, extra_pairs=()):
    rows = []
    cand = ck.get('PDP_cand')
    for m, g in GI_.items():
        mask = np.isin(np.arange(len(g)), cand) if (m == 'PDP' and cand is not None) else np.ones(len(g), bool)
        rows.append({'method': m, f'rho_vs_{ref_m}': spearmanr(g[mask], GI_[ref_m][mask]).correlation,
                     f'top20_shared_with_{ref_m}': len(set(top_genes(g)) & set(top_genes(GI_[ref_m]))),
                     'top5_genes': ', '.join(top_genes(g, k=5))})
    out = pd.DataFrame(rows)
    for a, b, lab in extra_pairs:
        out.loc[len(out)] = {'method': lab, f'rho_vs_{ref_m}': spearmanr(np.abs(ck[a]).mean(0), np.abs(ck[b]).mean(0)).correlation}
    return out


def sanity(GI_, GIr, cand=None):
    rows = []
    for m in GI_:
        if m not in GIr:
            continue
        mask = np.isin(np.arange(len(GI_[m])), cand) if (m == 'PDP' and cand is not None) else np.ones(len(GI_[m]), bool)
        rows.append({'method': m, 'sanity_abs_rho': abs(spearmanr(GI_[m][mask], GIr[m][mask]).correlation),
                     'sanity_top10_shared': len(set(np.argsort(-GI_[m])[:10]) & set(np.argsort(-GIr[m])[:10]))})
    return pd.DataFrame(rows)


def faith_insample(fn, X, ref, GI_, local_attr, T_, E_, tag, ref_m):
    ck = load(f'F_{tag}_faith')
    methods = list(GI_)
    if ck is None:
        S = make_stack(fn, X, {m: np.argsort(-GI_[m]) for m in methods}, 'resample', ref)
        point, boot = evaluate_stacks([(T_, E_, S)], len(methods))
        cp, sf = comp_suff(fn, X, local_attr, 'median', ref, fn(X).std())
        ck = {'point': point, 'boot': boot, 'comp': cp, 'suff': sf}
        save(f'F_{tag}_faith', ck)
    t = faith_table(ck['point'], ck['boot'], f'{tag}, resample fill', methods, ref_m)
    pc, ps = vs_ref_wilcoxon(ck['comp'], ref_m), vs_ref_wilcoxon(ck['suff'], ref_m)
    t['comp'] = [ck['comp'][m].mean() if m in ck['comp'] else np.nan for m in methods]
    t['comp_p_vs_ref'] = [pc.get(m, np.nan) for m in methods]
    t['suff'] = [ck['suff'][m].mean() if m in ck['suff'] else np.nan for m in methods]
    t['suff_p_vs_ref'] = [ps.get(m, np.nan) for m in methods]
    return t, ck


# ---------- (a) concatenation fusion ----------
sd = load('F_concat_model')
mc = ConcatFusion()
if sd is None:
    torch.manual_seed(42)
    mc = train_model(mc, X_all[:, :NC], X_all[:, NC:], T, E)
    save('F_concat_model', mc.state_dict())
else:
    mc.load_state_dict(sd); mc.eval()
Fc = Flat(mc).eval(); fnc = np_fn(Fc)
print(f"Concat fusion (full cohort) in-sample C = {harrell_c(T, E_int, fnc(X_all)):.4f}")
ckc = explain_all(Fc, X_all, X_all, T, E_int, 'concat')
GIc = gi_of(ckc, METHODS)
ckc_r = explain_all(Flat(randomise(mc, 0)).eval(), X_all, X_all, T, E_int, 'concat_rand')
GIc_r = gi_of(ckc_r, METHODS)
agr_c = agreement(GIc, ckc, 'SHAP-4096', [('SHAP-200', 'SHAP-200b', 'SHAP-200 re-run vs SHAP-200')])
san_c = sanity(GIc, GIc_r, ckc.get('PDP_cand'))
mod_c = {'exact': genomic_share_exact(fnc, X_all, NC),
         **{f'summed |{m}|': summed_share(ckc[m], NC) for m in ['IG', 'SHAP-4096', 'DeepLIFT', 'GradSHAP', 'LIME']}}
faith_c, _ = faith_insample(fnc, X_all, X_all, GIc, {**{m: ckc[m] for m in LOCAL_M}, 'PFI': GIc['PFI'], 'PDP': GIc['PDP']},
                            T, E_int, 'concat', 'IG')
print("\n(a) Concatenation fusion - agreement with converged KernelSHAP:"); print(agr_c.round(3).to_string(index=False))
print("\nSanity check (model randomisation):"); print(san_c.round(3).to_string(index=False))
print("\nGenomic share of the prediction:", {k: round(v, 3) for k, v in mod_c.items()})
print("\nFaithfulness (resample fill; per-patient comp/suff with median fill); p = Holm vs IG:")
print(faith_c.round(4).to_string(index=False))

# ---------- (b) random survival forest ----------
ck = load('F_rsf_models')
if ck is None:
    t0 = time.time(); rsf = fit_rsf(X_all, T, E_int)
    perm = np.random.RandomState(0).permutation(N)
    rsf_p = fit_rsf(X_all, T[perm], E_int[perm])
    ck = {'rsf': rsf, 'rsf_perm': rsf_p, 'perm': perm, 't_fit': time.time() - t0}
    save('F_rsf_models', ck)
rsf, rsf_p, perm = ck['rsf'], ck['rsf_perm'], ck['perm']
fn_rsf = lambda Z: rsf.predict(np.asarray(Z, float))
fn_rsfp = lambda Z: rsf_p.predict(np.asarray(Z, float))
t0 = time.time(); fn_rsf(np.repeat(X_all, 6, axis=0)[:1000]); rate = 1000 / (time.time() - t0)
print(f"\n(b) RSF ({RSF_TREES} trees) in-sample C = {harrell_c(T, E_int, fn_rsf(X_all)):.4f}; "
      f"~{rate:,.0f} predictions/s -> SHAP-{SHAP_HI} on {RSF_N} patients needs ~{SHAP_HI * RSF_BG * RSF_N / rate / 60:.0f} min")
RSF_IDX = np.sort(np.random.RandomState(9).choice(N, RSF_N, replace=False))
Xr = X_all[RSF_IDX]
ckr = load('F_rsf_expl') or {}
jobs = {'SHAP-200': lambda: kernel_shap(fn_rsf, Xr, X_all, SHAP_LO, seed=42, n_bg=RSF_BG),
        'SHAP-200b': lambda: kernel_shap(fn_rsf, Xr, X_all, SHAP_LO, seed=43, n_bg=RSF_BG),
        'SHAP-4096': lambda: kernel_shap(fn_rsf, Xr, X_all, SHAP_HI, seed=42, l1=False, n_bg=RSF_BG),
        'LIME': lambda: lime_local(fn_rsf, Xr, X_all, num_samples=300, seed=42),
        'PFI': lambda: pfi_global(fn_rsf, X_all, T, E_int, n_repeats=max(2, PFI_REPEATS // 2)),
        'perm_SHAP-4096': lambda: kernel_shap(fn_rsfp, Xr, X_all, SHAP_HI, seed=42, l1=False, n_bg=RSF_BG),
        'perm_LIME': lambda: lime_local(fn_rsfp, Xr, X_all, num_samples=300, seed=42),
        'perm_PFI': lambda: pfi_global(fn_rsfp, X_all, T[perm], E_int[perm], n_repeats=max(2, PFI_REPEATS // 2))}
for m, job in jobs.items():
    if m not in ckr:
        t0 = time.time(); ckr[m] = job(); ckr[f't_{m}'] = time.time() - t0
        save('F_rsf_expl', ckr); print(f"  RSF {m}: {ckr[f't_{m}']:.0f}s")
if 'PDP' not in ckr:
    cand = sorted(set().union(*[set(np.argsort(-np.abs(ckr[m]).mean(0))[:25]) for m in ['SHAP-4096', 'LIME']],
                              set(np.argsort(-ckr['PFI'])[:25])))
    ckr['PDP'] = pdp_global(fn_rsf, X_all, cand); ckr['PDP_cand'] = cand
    ckr['perm_PDP'] = pdp_global(fn_rsfp, X_all, cand)
    save('F_rsf_expl', ckr)
RSF_M = ['SHAP-200', 'SHAP-4096', 'LIME', 'PFI', 'PDP']
GIr = gi_of(ckr, RSF_M)
GIr_p = {m.replace('perm_', ''): (np.abs(ckr[m]).mean(0) if np.ndim(ckr[m]) == 2 else ckr[m])
         for m in ['perm_SHAP-4096', 'perm_LIME', 'perm_PFI', 'perm_PDP']}
agr_r = agreement(GIr, ckr, 'SHAP-4096', [('SHAP-200', 'SHAP-200b', 'SHAP-200 re-run vs SHAP-200')])
san_r = sanity(GIr, GIr_p, ckr['PDP_cand'])
mod_r = {'exact': genomic_share_exact(fn_rsf, Xr, NC, n_bg=RSF_BG), 'summed |SHAP-4096|': summed_share(ckr['SHAP-4096'], NC),
         'summed |LIME|': summed_share(ckr['LIME'], NC)}
faith_r, _ = faith_insample(fn_rsf, X_all, X_all, GIr, {}, T, E_int, 'rsf', 'SHAP-4096')
ckf = load('F_rsf_faith')
if 'comp' not in ckf or ckf['comp'].empty:
    cp, sf = comp_suff(fn_rsf, Xr, {m: ckr[m] for m in ['SHAP-200', 'SHAP-4096', 'LIME']}, 'median', X_all, fn_rsf(X_all).std())
    ckf['comp'], ckf['suff'] = cp, sf; save('F_rsf_faith', ckf)
pc, ps = vs_ref_wilcoxon(ckf['comp'], 'SHAP-4096'), vs_ref_wilcoxon(ckf['suff'], 'SHAP-4096')
faith_r['comp'] = [ckf['comp'][m].mean() if m in ckf['comp'] else np.nan for m in RSF_M]
faith_r['comp_p_vs_ref'] = [pc.get(m, np.nan) for m in RSF_M]
faith_r['suff'] = [ckf['suff'][m].mean() if m in ckf['suff'] else np.nan for m in RSF_M]
faith_r['suff_p_vs_ref'] = [ps.get(m, np.nan) for m in RSF_M]
print("\nRSF - agreement with converged KernelSHAP:"); print(agr_r.round(3).to_string(index=False))
print("\nRSF sanity check (forest refit on permuted labels):"); print(san_r.round(3).to_string(index=False))
print("\nRSF genomic share:", {k: round(v, 3) for k, v in mod_r.items()})
print(f"\nRSF faithfulness (resample fill; comp/suff on the {RSF_N} explained patients); p = Holm vs SHAP-4096:")
print(faith_r.round(4).to_string(index=False))
for nm, d in [('F4_concat_agreement', agr_c), ('F4_concat_sanity', san_c), ('F4_concat_faithfulness', faith_c),
              ('F4_rsf_agreement', agr_r), ('F4_rsf_sanity', san_r), ('F4_rsf_faithfulness', faith_r)]:
    d.to_csv(f'{OUT}/{nm}.csv', index=False)
pd.DataFrame([{'model': 'Concat fusion', **mod_c}, {'model': 'RSF', **mod_r}]).to_csv(f'{OUT}/F4_modality_shares.csv', index=False)
record('F4', {'concat': {'agreement': agr_c.to_dict('records'), 'sanity': san_c.to_dict('records'), 'modality': mod_c,
                         'faith': faith_c.to_dict('records')},
              'rsf': {'agreement': agr_r.to_dict('records'), 'sanity': san_r.to_dict('records'), 'modality': mod_r,
                      'faith': faith_r.to_dict('records')}})

In [ ]:
# ===== F5. Histologically confirmed PDAC only (n = 157): the key Table 5 metrics =====
# Same gated fusion architecture and training, refit on the PDAC subset (scalers fit on the subset). In-sample: agreement,
# low-budget SHAP noise, sanity check, exact vs summed modality shares, faithfulness (resample fill) and comp/suff.
# Out-of-fold: five fold models explain their own held-out PDAC patients (training-fold references, as Part E cell E4);
# comp/suff, deletion/insertion and fold stability, with equivalence vs IG.
if PDAC is None:
    raise FileNotFoundError("Part D histology file missing")
idx = np.where(PDAC)[0]
Xc_p, Xg_p, T_p, E_p = Xc_raw[idx], Xg_clip[idx], T[idx], E_int[idx]
sc_p, sg_p = RobustScaler().fit(Xc_p), RobustScaler().fit(Xg_p)
X_p = np.hstack([sc_p.transform(Xc_p), sg_p.transform(Xg_p)]).astype(f32)
sd = load('F_pdac_model')
mp = GatedFusion()
if sd is None:
    torch.manual_seed(42); mp = train_model(mp, X_p[:, :NC], X_p[:, NC:], T_p, E_p.astype(f32)); save('F_pdac_model', mp.state_dict())
else:
    mp.load_state_dict(sd); mp.eval()
Fp = Flat(mp).eval(); fnp = np_fn(Fp)
print(f"PDAC model: n = {len(idx)}, in-sample C = {harrell_c(T_p, E_p, fnp(X_p)):.4f}")
ckp = explain_all(Fp, X_p, X_p, T_p, E_p, 'pdac')
GIp = gi_of(ckp, METHODS)
ckp_r = explain_all(Flat(randomise(mp, 0)).eval(), X_p, X_p, T_p, E_p, 'pdac_rand', hi=False)
GIp_r = gi_of(ckp_r, METHODS)
agr_p = agreement(GIp, ckp, 'IG', [('SHAP-200', 'SHAP-200b', 'SHAP-200 re-run vs SHAP-200')])
san_p = sanity(GIp, GIp_r, ckp.get('PDP_cand'))
mod_p = {'exact': genomic_share_exact(fnp, X_p, NC),
         **{f'summed |{m}|': summed_share(ckp[m], NC) for m in ['IG', 'SHAP-4096', 'DeepLIFT', 'GradSHAP', 'LIME']}}
faith_p, ckpf = faith_insample(fnp, X_p, X_p, GIp, {**{m: ckp[m] for m in LOCAL_M}, 'PFI': GIp['PFI'], 'PDP': GIp['PDP']},
                               T_p, E_p, 'pdac', 'IG')

# Out-of-fold on the PDAC subset
FOLDS_P = list(StratifiedKFold(5, shuffle=True, random_state=42).split(Xc_p, E_p))
fold_ck = load('F_pdac_folds') or {}
for f, (tr, te) in enumerate(FOLDS_P):
    if f in fold_ck and all(m in fold_ck[f]['local'] for m in LOCAL_M) and 'PDP' in fold_ck[f]:
        continue
    rec = fold_ck.get(f) or {'local': {}}
    sc, sg = RobustScaler().fit(Xc_p[tr]), RobustScaler().fit(Xg_p[tr])
    X_tr = np.hstack([sc.transform(Xc_p[tr]), sg.transform(Xg_p[tr])]).astype(f32)
    X_te = np.hstack([sc.transform(Xc_p[te]), sg.transform(Xg_p[te])]).astype(f32)
    if 'state' not in rec:
        torch.manual_seed(42)
        m_ = train_model(GatedFusion(), X_tr[:, :NC], X_tr[:, NC:], T_p[tr], E_p[tr].astype(f32))
        rec['state'] = m_.state_dict()
    m_ = GatedFusion(); m_.load_state_dict(rec['state']); m_.eval()
    F_ = Flat(m_).eval(); fn_ = np_fn(F_); med_tr = np.median(X_tr, 0).astype(f32)
    jobs = {'SHAP-200': lambda: kernel_shap(fn_, X_te, X_tr, SHAP_LO, seed=42),
            'SHAP-4096': lambda: kernel_shap(fn_, X_te, X_tr, SHAP_HI, seed=42, l1=False),
            'LIME': lambda: lime_local(fn_, X_te, X_tr, num_samples=300, seed=42),
            'IG': lambda: ig_attr(F_, X_te, med_tr),
            'GradSHAP': lambda: gradshap_attr(F_, X_te, shap.sample(X_tr, 30), seed=42),
            'DeepLIFT': lambda: deeplift_attr(F_, X_te, med_tr)[0]}
    for m, job in jobs.items():
        if m not in rec['local']:
            rec['local'][m] = job(); fold_ck[f] = rec; save('F_pdac_folds', fold_ck)
    if 'PFI' not in rec:
        rec['PFI'] = pfi_global(fn_, X_tr, T_p[tr], E_p[tr], n_repeats=max(2, PFI_REPEATS // 2))
    if 'PDP' not in rec:
        g = {m: np.abs(rec['local'][m]).mean(0) for m in ['SHAP-200', 'LIME', 'IG']}
        cand = sorted(set().union(*[set(np.argsort(-v)[:25]) for v in g.values()], set(np.argsort(-rec['PFI'])[:25])))
        rec['PDP'] = pdp_global(fn_, X_tr, cand)
    rec['X_tr'], rec['X_te'] = X_tr, X_te
    fold_ck[f] = rec; save('F_pdac_folds', fold_ck)
    print(f"  PDAC fold {f + 1}/5 done")

comp_l, suff_l, parts = [], [], []
for f, (tr, te) in enumerate(FOLDS_P):
    rec = fold_ck[f]
    m_ = GatedFusion(); m_.load_state_dict(rec['state']); m_.eval(); fn_ = np_fn(Flat(m_).eval())
    attrs = {**{m: rec['local'][m] for m in LOCAL_M}, 'PFI': rec['PFI'], 'PDP': rec['PDP']}
    cp, sf = comp_suff(fn_, rec['X_te'], attrs, 'median', rec['X_tr'], fn_(rec['X_tr']).std())
    comp_l.append(cp); suff_l.append(sf)
    gi_f = {m: (np.abs(attrs[m]).mean(0) if np.ndim(attrs[m]) == 2 else attrs[m]) for m in METHODS}
    parts.append((T_p[te], E_p[te], make_stack(fn_, rec['X_te'], {m: np.argsort(-gi_f[m]) for m in METHODS},
                                               'median', rec['X_tr'], seed=42 + f)))
    rec['GI'] = gi_f
ck = load('F_pdac_oof_delins')
if ck is None:
    point, boot = evaluate_stacks(parts, len(METHODS)); ck = {'point': point, 'boot': boot}; save('F_pdac_oof_delins', ck)
oof_tab = faith_table(ck['point'], ck['boot'], 'PDAC OOF, median fill', METHODS, 'IG')
print(f"  held-out C of the PDAC fold models (mean): {ck['point']['c0']:.4f}")
comp_o, suff_o = pd.concat(comp_l, ignore_index=True), pd.concat(suff_l, ignore_index=True)
eq = []
for metric, W in [('comp', comp_o), ('suff', suff_o)]:
    for m in METHODS:
        if m != 'IG':
            pnt, lo, hi = boot_diff_ci(W[m].values, W['IG'].values)
            eq.append({'metric': f'{metric} OOF', 'method': m, 'mean': W[m].mean(), 'IG_mean': W['IG'].mean(), 'diff_vs_IG': pnt,
                       'ci90_lo': lo, 'ci90_hi': hi, 'equivalent': bool(lo > -MARGIN[metric] and hi < MARGIN[metric])})
stab = []
for m in METHODS:
    G = [fold_ck[f]['GI'][m] for f in range(5)]
    rho = np.nanmean([spearmanr(G[a], G[b]).correlation for a, b in combinations(range(5), 2)]) if m != 'PDP' else np.nan
    stab.append({'method': m, 'fold_rank_rho': rho,
                 'fold_jaccard10': np.mean([jaccard(np.argsort(-G[a])[:10], np.argsort(-G[b])[:10]) for a, b in combinations(range(5), 2)])})
print("\nPDAC in-sample - agreement with IG:"); print(agr_p.round(3).to_string(index=False))
print("\nPDAC sanity check:"); print(san_p.round(3).to_string(index=False))
print("\nPDAC genomic share:", {k: round(v, 3) for k, v in mod_p.items()})
print("\nPDAC in-sample faithfulness; p = Holm vs IG:"); print(faith_p.round(4).to_string(index=False))
print("\nPDAC out-of-fold deletion / insertion:"); print(oof_tab.round(4).to_string(index=False))
print("\nPDAC out-of-fold comp / suff, equivalence vs IG:"); print(pd.DataFrame(eq).round(4).to_string(index=False))
print("\nPDAC fold stability:"); print(pd.DataFrame(stab).round(3).to_string(index=False))
for nm, d in [('F5_pdac_agreement', agr_p), ('F5_pdac_sanity', san_p), ('F5_pdac_faith_insample', faith_p),
              ('F5_pdac_oof_delins', oof_tab), ('F5_pdac_oof_comp_suff_equivalence', pd.DataFrame(eq)),
              ('F5_pdac_stability', pd.DataFrame(stab))]:
    d.to_csv(f'{OUT}/{nm}.csv', index=False)
pd.DataFrame([mod_p]).to_csv(f'{OUT}/F5_pdac_modality.csv', index=False)
record('F5_pdac', {'agreement': agr_p.to_dict('records'), 'sanity': san_p.to_dict('records'), 'modality': mod_p,
                   'faith_insample': faith_p.to_dict('records'), 'oof_delins': oof_tab.to_dict('records'),
                   'oof_comp_suff': eq, 'stability': stab})

In [ ]:
# ===== F6. More clinical variables: does the clinical share stay at ~16% with richer clinical input? =====
# Adds sex, histological grade, residual disease (R0/R1/R2) and nodal status (N0/N1) from the GDC (public API) to age and
# stage. Missing values -> training-set median (counts printed). Compares 5-fold out-of-fold C (same folds as v5) of the
# clinical network, the gated fusion network and a clinical Cox model with 2 vs extended clinical inputs, and the exact
# modality Shapley share of the full-cohort gated network.
EXTRA_CSV = f'{DATA_DIR}/gdc_clinical_extra.csv'


def gdc_extra():
    if os.path.exists(EXTRA_CSV):
        return pd.read_csv(EXTRA_CSV)
    filt = {"op": "in", "content": {"field": "project.project_id", "value": ["TCGA-PAAD"]}}
    fields = ','.join(['submitter_id', 'demographic.gender', 'diagnoses.tumor_grade', 'diagnoses.residual_disease',
                       'diagnoses.ajcc_pathologic_n', 'diagnoses.primary_diagnosis'])
    r = requests.get('https://api.gdc.cancer.gov/cases', timeout=120,
                     params={'filters': json.dumps(filt), 'fields': fields, 'format': 'json', 'size': '1000'})
    r.raise_for_status()
    rows = []
    for h in r.json()['data']['hits']:
        dx = [d for d in (h.get('diagnoses') or []) if d.get('primary_diagnosis')] or (h.get('diagnoses') or [{}])
        d0 = dx[0] if dx else {}
        rows.append({'PATIENT_ID': h['submitter_id'], 'gender': (h.get('demographic') or {}).get('gender'),
                     'tumor_grade': d0.get('tumor_grade'), 'residual_disease': d0.get('residual_disease'),
                     'ajcc_pathologic_n': d0.get('ajcc_pathologic_n')})
    out = pd.DataFrame(rows)
    out.to_csv(EXTRA_CSV, index=False)
    return out


try:
    clin = gdc_extra().drop_duplicates('PATIENT_ID').set_index('PATIENT_ID').reindex(PID)
except Exception as e:
    clin = None
    print(f"GDC clinical download failed ({e}); F6 skipped.")

if clin is not None:
    def enc_grade(s):
        m = re.search(r'G([1-4])', str(s).upper()); return float(m.group(1)) if m else np.nan

    def enc_resid(s):
        m = re.search(r'R([0-2])', str(s).upper()); return float(m.group(1)) if m else np.nan

    def enc_n(s):
        m = re.search(r'N([0-3])', str(s).upper()); return float(m.group(1) != '0') if m else np.nan

    extra = pd.DataFrame({'SEX_MALE': (clin['gender'].astype(str).str.lower() == 'male').astype(float).where(clin['gender'].notna()),
                          'GRADE': clin['tumor_grade'].map(enc_grade), 'RESIDUAL': clin['residual_disease'].map(enc_resid),
                          'NODE_POS': clin['ajcc_pathologic_n'].map(enc_n)})
    print("Missing values (filled with the training-fold median):", extra.isna().sum().to_dict())
    print("Distributions:", {c: extra[c].value_counts(dropna=True).sort_index().to_dict() for c in extra})
    keep = [c for c in extra if extra[c].notna().mean() >= 0.5 and extra[c].nunique() > 1]
    XC_EXT_RAW = np.column_stack([Xc_raw, extra[keep].values]).astype(f32)
    CLIN_EXT = CLIN_COLS + keep
    print(f"Extended clinical inputs: {CLIN_EXT}")

    def impute(Xtr, Xte):
        med = np.nanmedian(Xtr, 0)
        return np.where(np.isnan(Xtr), med, Xtr).astype(f32), np.where(np.isnan(Xte), med, Xte).astype(f32)

    def cv_models(Xc_in):
        res = {'Clinical MLP': [], 'Gated fusion': [], 'CoxPH (clinical)': []}
        for tr, te in FOLDS:
            a_tr, a_te = impute(Xc_in[tr], Xc_in[te])
            sc, sg = RobustScaler().fit(a_tr), RobustScaler().fit(Xg_clip[tr])
            Xc_tr, Xc_te = sc.transform(a_tr).astype(f32), sc.transform(a_te).astype(f32)
            Xg_tr, Xg_te = sg.transform(Xg_clip[tr]).astype(f32), sg.transform(Xg_clip[te]).astype(f32)
            for name, cls in [('Clinical MLP', ClinicalMLP), ('Gated fusion', GatedFusion)]:
                torch.manual_seed(42)
                m_ = train_model(cls(Xc_in.shape[1], Xg_clip.shape[1]), Xc_tr, Xg_tr, T[tr], E[tr])
                res[name].append(harrell_c(T[te], E_int[te], risk_of(m_, Xc_te, Xg_te)))
            d = pd.DataFrame(Xc_tr, columns=[f'c{i}' for i in range(Xc_tr.shape[1])]); d['T'] = T[tr]; d['E'] = E[tr]
            cph = CoxPHFitter(penalizer=0.01).fit(d, 'T', 'E')
            res['CoxPH (clinical)'].append(harrell_c(T[te], E_int[te], cph.predict_log_partial_hazard(
                pd.DataFrame(Xc_te, columns=[f'c{i}' for i in range(Xc_te.shape[1])])).values))
        return res

    ck = load('F6_cv')
    if ck is None:
        ck = {'2 vars': cv_models(Xc_raw), 'extended': cv_models(XC_EXT_RAW)}
        save('F6_cv', ck)
    rows = []
    for setting, res in ck.items():
        for mname, cs in res.items():
            rows.append({'clinical inputs': setting, 'model': mname, 'OOF_C_mean': np.mean(cs), 'OOF_C_sd': np.std(cs)})
    F6a = pd.DataFrame(rows)

    # Full-cohort gated fusion with extended clinical inputs: exact modality share and IG ranks of the clinical features
    ck2 = load('F6_full')
    if ck2 is None:
        a_all, _ = impute(XC_EXT_RAW, XC_EXT_RAW)
        sc = RobustScaler().fit(a_all)
        X_ext = np.hstack([sc.transform(a_all), X_all[:, NC:]]).astype(f32)
        torch.manual_seed(42)
        m_ext = train_model(GatedFusion(len(CLIN_EXT), 1000), X_ext[:, :len(CLIN_EXT)],
                            X_ext[:, len(CLIN_EXT):], T, E)
        ck2 = {'state': m_ext.state_dict(), 'X_ext': X_ext}
        save('F6_full', ck2)
    m_ext = GatedFusion(len(CLIN_EXT), 1000); m_ext.load_state_dict(ck2['state']); m_ext.eval()
    X_ext = ck2['X_ext']
    F_ext = Flat(m_ext, nc=len(CLIN_EXT)).eval(); fn_ext = np_fn(F_ext)
    ig_ext = ig_attr(F_ext, X_ext, np.median(X_ext, 0))
    share_ext = genomic_share_exact(fn_ext, X_ext, len(CLIN_EXT))
    share_2 = genomic_share_exact(pred_full, X_all, NC)
    ranks = rankdata(-np.abs(ig_ext).mean(0), method='min')
    F6b = pd.DataFrame({'feature': CLIN_EXT, 'IG_rank_of_all_inputs': ranks[:len(CLIN_EXT)].astype(int),
                        'mean_abs_IG': np.abs(ig_ext).mean(0)[:len(CLIN_EXT)]})
    print(F6a.round(4).to_string(index=False))
    print(f"\nExact genomic share: 2 clinical inputs {share_2:.3f} -> extended clinical inputs {share_ext:.3f} "
          f"(clinical share {1 - share_2:.3f} -> {1 - share_ext:.3f}); summed |IG| genomic share (extended): "
          f"{summed_share(ig_ext, len(CLIN_EXT)):.3f}")
    print(F6b.round(4).to_string(index=False))
    F6a.to_csv(f'{OUT}/F6_clinical_cv.csv', index=False); F6b.to_csv(f'{OUT}/F6_clinical_ig_ranks.csv', index=False)
    record('F6_clinical', {'inputs': CLIN_EXT, 'missing': extra.isna().sum().to_dict(), 'cv': F6a.to_dict('records'),
                           'genomic_share_2vars': share_2, 'genomic_share_extended': share_ext,
                           'ig_ranks': F6b.to_dict('records')})

In [ ]:
# ===== F7. Summary =====
with open(f'{OUT}/partF_results.json', 'w') as fh:
    json.dump(RESULTS, fh, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
print(f"Saved {OUT}/partF_results.json")
print("Send back the zipped partF folder (or at least partF_results.json, all F*.csv files and fig_moffitt_km.png).")